In [5]:
!pip install librosa numpy scikit-learn tensorflow joblib

  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached tensorboard-2.19.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
   ---------------------------------------- 0.0/376.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/376.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/376.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/376.0 MB ? eta -:--:--
   ----------

In [6]:
import os
import glob
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.sequence import pad_sequences
import joblib

# Define constants for audio processing
SAMPLE_RATE = 44100
NUM_MFCC = 13
MAX_LEN = 500  # Maximum length of audio features

# Function to preprocess the audio clip
def preprocess_audio(audio_path):
    # Load the audio file
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

    # Normalize the audio to have maximum amplitude of 1
    normalized_audio = audio / np.max(np.abs(audio))

    return normalized_audio

# Function to extract MFCC features from audio data
def extract_mfcc(audio_data):
    mfcc_features = librosa.feature.mfcc(y=audio_data, sr=SAMPLE_RATE, n_mfcc=NUM_MFCC)

    return mfcc_features.T

# Function to extract features and labels from the dataset folder
def extract_features_and_labels(dataset_folder):
    features = []
    labels = []

    # Iterate through subfolders (classes) in the dataset folder
    for label_folder in os.listdir(dataset_folder):
        label_path = os.path.join(dataset_folder, label_folder)

        # Iterate through audio files in each label folder
        for audio_path in glob.glob(os.path.join(label_path, '*.wav')):
            label = label_folder

            # Preprocess the audio clip
            preprocessed_audio = preprocess_audio(audio_path)

            # Extract MFCC features
            mfcc_features = extract_mfcc(preprocessed_audio)

            # Append the features and label to the lists
            features.append(mfcc_features)
            labels.append(label)

    return features, labels

# Path to the dataset folder
dataset_folder = 'NEW_AUDIO_DATA'

# Extract features and labels from the dataset folder
features, labels = extract_features_and_labels(dataset_folder)

# Pad or truncate the features to a fixed length
features_padded = pad_sequences(features, maxlen=MAX_LEN, padding='post', truncating='post', dtype='float32')

# Convert the lists to numpy arrays
features_padded = np.array(features_padded)
labels = np.array(labels)

# Flatten the features array to match the expected shape
features_flattened = features_padded.reshape(features_padded.shape[0], -1)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features_flattened, labels, test_size=0.2, random_state=42)

# Create a random forest classifier
rf_classifier = RandomForestClassifier()

# Train the random forest classifier
rf_classifier.fit(X_train, y_train)
joblib.dump(rf_classifier, 'vocal_training_model.joblib')

# Predict the labels for the test set
y_pred = rf_classifier.predict(X_test)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy of the model
print("Accuracy:", accuracy)


C:\Users\Kavindu Kanchana\Anaconda\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Kavindu Kanchana\Anaconda\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Kavindu Kanchana\Anaconda\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Accuracy: 0.8166666666666667


In [8]:
!pip install flask-cors

  Using cached flask_cors-6.0.1-py3-none-any.whl.metadata (5.3 kB)
Using cached flask_cors-6.0.1-py3-none-any.whl (13 kB)


In [9]:
from flask import Flask, jsonify, request
from flask_cors import CORS, cross_origin
import joblib

In [10]:
app = Flask(__name__)
cors = CORS(app)
app.config['CORS_HEADERS'] = 'Content-Type'

In [11]:
# Load the trained model from the file
rf_classifier = joblib.load("vocal_training_model.joblib")

In [12]:
@app.route('/api/checkVoice', methods=['POST'])
def check_voice():
    audio_file = request.files['fileData']
    label = request.form.get('label')
    
    # Load the pre-recorded audio clip
    preprocessed_audio = preprocess_audio(audio_file)

    # Extract MFCC features
    mfcc_features = extract_mfcc(preprocessed_audio)

    # Pad or truncate the features to a fixed length
    features_padded = pad_sequences([mfcc_features], maxlen=MAX_LEN, padding='post', truncating='post', dtype='float32')

    # Flatten the features array to match the expected shape
    features_flattened = features_padded.reshape(1, -1)

    # Predict the label using the trained random forest classifier
    predicted_label = rf_classifier.predict(features_flattened)

    isValid = False
    
    print(predicted_label)
    
    if label==predicted_label:
        isValid = True
    
    # Logic to fetch data
    data = {'isValid': isValid}
    return jsonify(data)

In [ ]:
if __name__ == '__main__':
    app.run(port=8050) 

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:8050
Press CTRL+C to quit
